# Cenário 6: Correlação entre a Fortuna de Musk e a População de Rua no Brasil

## Contexto
Enquanto a fortuna de Elon Musk cresceu de forma exponencial na última década, a população em situação de rua no Brasil também apresentou crescimento contínuo. Analisamos a correlação estatística entre esses dois fenômenos.

> ⚠️ **Nota metodológica:** Correlação estatística não implica relação causal direta. Ambos os fenômenos são multifatoriais e refletem tendências macroeconômicas globais.

**Fontes:**
- [IPEA – População em Situação de Rua](https://www.ipea.gov.br)
- [OBPopRua/UFMG](https://obpoprua.direito.ufmg.br)
- [Bloomberg Billionaires Index](https://www.bloomberg.com/billionaires/)
- [Forbes Billionaires](https://www.forbes.com/real-time-billionaires/)

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

MUSK_USD   = 1_000_000_000_000   # US$ 1 trilhão (Forbes, jun/2026)
MUSK_BRL   = MUSK_USD * 5.71    # câmbio aproximado jun/2026
USD_BRL    = 5.71
DATA       = 'data/'
rua  = pd.read_csv(DATA + 'historico_pop_rua_brasil.csv')
musk_h = pd.read_csv(DATA + 'historico_fortuna_musk.csv')

# Pegar último valor por ano para Musk
musk_ano = musk_h.groupby('ano')['fortuna_bilhoes_usd'].last().reset_index()
df = pd.merge(rua, musk_ano, on='ano', how='inner')

correlacao = np.corrcoef(df['fortuna_bilhoes_usd'], df['populacao_rua'])[0, 1]
print(f"Anos em comum: {list(df['ano'])}")
print(f"Correlação de Pearson: {correlacao:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Gráfico 1: Série temporal duplo eixo
ax1 = axes[0]
ax2 = ax1.twinx()
l1, = ax1.plot(df['ano'], df['populacao_rua']/1000, 'o-', color='#c0392b', linewidth=2.5,
               markersize=8, label='Pop. em situação de rua (mil pessoas)')
l2, = ax2.plot(df['ano'], df['fortuna_bilhoes_usd'], 's--', color='#2980b9', linewidth=2.5,
               markersize=8, label='Fortuna de Musk (bilhões USD)')
ax1.set_xlabel('Ano', fontsize=11)
ax1.set_ylabel('Pop. em situação de rua (mil pessoas)', color='#c0392b', fontsize=11)
ax2.set_ylabel('Fortuna de Musk (bilhões USD)', color='#2980b9', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#c0392b')
ax2.tick_params(axis='y', labelcolor='#2980b9')
ax1.set_title('Evolução da Fortuna de Musk\ne da Pop. em Situação de Rua no Brasil', fontsize=12, fontweight='bold')
lines = [l1, l2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=9)

# Gráfico 2: Scatter com regressão
ax = axes[1]
ax.scatter(df['fortuna_bilhoes_usd'], df['populacao_rua']/1000, color='#e74c3c', s=120, zorder=5)
for _, row in df.iterrows():
    ax.annotate(str(row['ano']), (row['fortuna_bilhoes_usd'], row['populacao_rua']/1000),
                textcoords='offset points', xytext=(5, 5), fontsize=9)

z = np.polyfit(df['fortuna_bilhoes_usd'], df['populacao_rua']/1000, 1)
p = np.poly1d(z)
x_line = np.linspace(df['fortuna_bilhoes_usd'].min(), df['fortuna_bilhoes_usd'].max(), 100)
ax.plot(x_line, p(x_line), '--', color='#2980b9', linewidth=2, alpha=0.7, label=f'Tendência (r={correlacao:.2f})')
ax.set_xlabel('Fortuna de Musk (bilhões USD)', fontsize=11)
ax.set_ylabel('Pop. em situação de rua no Brasil (mil)', fontsize=11)
ax.set_title(f'Correlação entre Riqueza de Musk\ne Desigualdade Social no Brasil\n(r = {correlacao:.3f})',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

axes[1].text(0.98, -0.08, 'Fonte: IPEA/OBPopRua | Forbes/Bloomberg',
             transform=axes[1].transAxes, fontsize=8, ha='right', color='gray')

plt.suptitle('Correlação: Crescimento da Fortuna de Musk x Crescimento da População de Rua no Brasil',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA + 'output_06_correlacao_rua.png', dpi=150, bbox_inches='tight')
plt.show()


Anos em comum: [2012, 2015, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Correlação de Pearson: 0.8742


In [2]:
print("=" * 65)
print("ANÁLISE: CORRELAÇÃO FORTUNA MUSK x POP. RUA BRASIL")
print("=" * 65)
print(f"\nCoeficiente de correlação de Pearson: {correlacao:.4f}")
if correlacao > 0.9:
    interpretacao = "correlação positiva muito forte"
elif correlacao > 0.7:
    interpretacao = "correlação positiva forte"
else:
    interpretacao = "correlação positiva moderada"
print(f"Interpretação: {interpretacao}")
print(f"\nAnos analisados: {list(df['ano'])}")
print(f"\nEvolução da população de rua:")
for _, row in df.iterrows():
    print(f"  {int(row['ano'])}: {int(row['populacao_rua']):>7,} pessoas | Musk: US$ {row['fortuna_bilhoes_usd']:>6,.0f}bi")
crescimento_rua = (df['populacao_rua'].iloc[-1] / df['populacao_rua'].iloc[0] - 1) * 100
crescimento_musk = (df['fortuna_bilhoes_usd'].iloc[-1] / df['fortuna_bilhoes_usd'].iloc[0] - 1) * 100
print(f"\nCrescimento da pop. de rua:     {crescimento_rua:.0f}%")
print(f"Crescimento da fortuna de Musk: {crescimento_musk:.0f}%")
print("\n⚠️  ATENÇÃO: Correlação não implica causalidade direta.")
print("   Ambos os fenômenos são reflexos de fatores macroeconômicos")
print("   e estruturais mais amplos, como concentração de capital,")
print("   pandemia de COVID-19, crise econômica e desigualdade sistêmica.")
print("\nFontes: IPEA (2023) | OBPopRua/UFMG | Forbes | Bloomberg")


ANÁLISE: CORRELAÇÃO FORTUNA MUSK x POP. RUA BRASIL

Coeficiente de correlação de Pearson: 0.8742
Interpretação: correlação positiva forte

Anos analisados: [2012, 2015, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Evolução da população de rua:
  2012:  94,163 pessoas | Musk: US$      2bi
  2015: 101,854 pessoas | Musk: US$     13bi
  2019: 168,838 pessoas | Musk: US$     20bi
  2020: 190,052 pessoas | Musk: US$    165bi
  2021: 234,000 pessoas | Musk: US$    244bi
  2022: 281,472 pessoas | Musk: US$    137bi
  2023: 300,000 pessoas | Musk: US$    232bi
  2024: 327,000 pessoas | Musk: US$    432bi
  2025: 365,822 pessoas | Musk: US$    647bi

Crescimento da pop. de rua:     288%
Crescimento da fortuna de Musk: 32250%

⚠️  ATENÇÃO: Correlação não implica causalidade direta.
   Ambos os fenômenos são reflexos de fatores macroeconômicos
   e estruturais mais amplos, como concentração de capital,
   pandemia de COVID-19, crise econômica e desigualdade sistêmica.

Fontes: IPEA (2023) | OBPopRu